In [ ]:
import glob
import json
import random
from pathlib import Path
from torch.utils.data import DataLoader

# ==========================================
# 1. Helper function to resolve .pt paths
# ==========================================
def get_pt_file_paths(selected_blocks, lookup_dict, preprocessed_dirs):
    """Gathers full file paths for all .pt tiles belonging to selected_blocks."""
    # Build a quick search map of filename -> full path across all split directories
    pt_path_map = {}
    for prep_dir in preprocessed_dirs:
        for pt_path in Path(prep_dir).glob("*.pt"):
            pt_path_map[pt_path.name] = str(pt_path)

    # Resolve full paths for chosen blocks
    resolved_paths = []
    for block in selected_blocks:
        pt_filenames = lookup_dict.get(block, [])
        for fname in pt_filenames:
            if fname in pt_path_map:
                resolved_paths.append(pt_path_map[fname])
            else:
                print(f"Warning: {fname} from block {block} not found in preprocessed directories.")

    return sorted(resolved_paths)


# ==========================================
# 2. Setup Directories and Load Lookup Table
# ==========================================
preprocessed_dirs = [
    "E:/preprocessed_train",
    "E:/preprocessed_val",
    "E:/preprocessed_test",
]

# Load the lookup table created in the previous step
with open("block_tile_lookup.json", "r") as f:
    lookup_table = json.load(f)

# All available blocks
all_blocks = list(lookup_table.keys())  # ['block1', 'block2', ..., 'block10']


# ==========================================
# 3. Flexible Block Selection / Splitting
# ==========================================
# Option A: Manual explicit assignment
train_blocks = ["block1", "block2", "block3", "block4", "block5", "block6", "block7", "block8"]
val_blocks = ["block9"]
test_blocks = ["block10"]

# Option B: Random split (uncomment if you want randomized block assignments)
# random.seed(42)
# shuffled_blocks = all_blocks.copy()
# random.shuffle(shuffled_blocks)
# train_blocks = shuffled_blocks[:8]
# val_blocks = [shuffled_blocks[8]]
# test_blocks = [shuffled_blocks[9]]


# ==========================================
# 4. Resolve File Paths per Split
# ==========================================
train_pt_paths = get_pt_file_paths(train_blocks, lookup_table, preprocessed_dirs)
val_pt_paths = get_pt_file_paths(val_blocks, lookup_table, preprocessed_dirs)
test_pt_paths = get_pt_file_paths(test_blocks, lookup_table, preprocessed_dirs)

print(f"Train tiles: {len(train_pt_paths)} | Val tiles: {len(val_pt_paths)} | Test tiles: {len(test_pt_paths)}")


# ==========================================
# 5. Instantiate Datasets & DataLoaders
# ==========================================
train_dataset = FastPreprocessedDataset(
    pt_file_paths=train_pt_paths, 
    transform=get_train_transforms()
)

val_dataset = FastPreprocessedDataset(
    pt_file_paths=val_pt_paths, 
    transform=None
)

test_dataset = FastPreprocessedDataset(
    pt_file_paths=test_pt_paths, 
    transform=None
)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=0, pin_memory=True)